# N03 — Benchmarks: Random Walk, RPPP, CIRP, encuestas BCU

**Objetivo:** correr únicamente modelos *benchmark* para estimar USD/UYU a distintos horizontes, **evaluando en niveles (USD/UYU)**, guardando:
- métricas por horizonte (MAE/RMSE/R2),
- predicciones OOS en niveles con **fecha, fecha_objetivo, dolar_t, dolar_pred_tH, dolar_real_tH**,
- gráficos por modelo y horizonte,
- profiling de **tiempo** y **memoria** por modelo.


## 1) Imports + configuración general


In [2]:
import os
import warnings
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")



# -------------------------
# Config
# -------------------------
DATE_COL   = "Fecha"
TARGET_COL = "usd_dlog"   # retornos log diarios del USD/UYU (dlog)

HORIZONS = {"1m": 21, "3m": 63, "6m": 126, "12m": 252}
HORIZON_MONTHS = {"1m": 1, "3m": 3, "6m": 6, "12m": 12}

# Aproximación de disponibilidad de IPC (para evitar look-ahead)
# Se puede ajustar si luego quieres usar el calendario exacto de publicaciones.
UY_CPI_RELEASE_DAY = 5    # IPC Uruguay: aprox. día 5 del mes siguiente
US_CPI_RELEASE_DAY = 14   # CPI USA: aprox. día 14 del mes siguiente

BASE_DIR = Path.cwd()
# Inputs
INPUT_DAILY_FILE  = BASE_DIR / "results_n00" /"df_daily.xlsx"
INPUT_IPC_UY_FILE = BASE_DIR / "results_n03" /"ipc_mensual_uy.xlsx"
INPUT_IPC_US_FILE = BASE_DIR / "results_n03" /"ipc_usa_mensual_fred.xlsx"
INPUT_UST_FILE = BASE_DIR / "results_n03" /"treasuries_us_curve_daily.xlsx"
INPUT_LRM_FILE = BASE_DIR / "results_n03" /"tasas_LRM_plazo_ref.xlsx"
INPUT_BCU_FILE    = BASE_DIR / "results_n03" /"expectativas_BCU.xlsx"   # mensual: Fecha, 1 mes, 6 meses, 12 meses

# Outputs (separado de ML)
BASE_SAVE_DIR = "results_n03"
OOS_DIR   = os.path.join(BASE_SAVE_DIR, "oos")
PLOTS_DIR = os.path.join(BASE_SAVE_DIR, "plots")

os.makedirs(BASE_SAVE_DIR, exist_ok=True)
os.makedirs(OOS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print("BASE_SAVE_DIR:", BASE_SAVE_DIR)
print("OOS_DIR:", OOS_DIR)
print("PLOTS_DIR:", PLOTS_DIR)


BASE_SAVE_DIR: results_n03
OOS_DIR: results_n03\oos
PLOTS_DIR: results_n03\plots


## 2) Carga de datos + alineación de fechas


In [3]:
def _check_file(path: str):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"No se encontró el archivo '{path}'.")

# -------------------------
# Verificar archivos
# -------------------------
_check_file(INPUT_DAILY_FILE)
_check_file(INPUT_BCU_FILE)
_check_file(INPUT_IPC_UY_FILE)
_check_file(INPUT_IPC_US_FILE)
_check_file(INPUT_UST_FILE)
_check_file(INPUT_LRM_FILE)

# -------------------------
# Cargar base diaria principal
# -------------------------
df_daily = pd.read_excel(INPUT_DAILY_FILE)

# -------------------------
# Validaciones
# -------------------------
if DATE_COL not in df_daily.columns:
    raise ValueError(
        f"df_daily no tiene '{DATE_COL}'. Columnas: {list(df_daily.columns)[:30]} ..."
    )

if TARGET_COL not in df_daily.columns:
    raise ValueError(
        f"df_daily no tiene '{TARGET_COL}'. Columnas: {list(df_daily.columns)[:30]} ..."
    )

# -------------------------
# Limpieza de fechas
# -------------------------
df_daily[DATE_COL] = pd.to_datetime(df_daily[DATE_COL], errors="coerce")

df_daily = (
    df_daily
    .dropna(subset=[DATE_COL])
    .sort_values(DATE_COL)
    .reset_index(drop=True)
)

print("df_daily:", df_daily.shape)
print("Periodo:", df_daily[DATE_COL].min(), "->", df_daily[DATE_COL].max())

df_daily: (2643, 125)
Periodo: 2015-01-05 00:00:00 -> 2025-09-05 00:00:00


## 3) Preparar `dfp` (paridades / niveles) y checks mínimos


In [4]:
dfp = df_daily.copy()

dfp[DATE_COL] = pd.to_datetime(dfp[DATE_COL], errors="coerce")

dfp = (
    dfp
    .dropna(subset=[DATE_COL])
    .sort_values(DATE_COL)
    .set_index(DATE_COL)
)

# -------------------------
# Validar columnas mínimas
# -------------------------

required_cols = ["dolar", TARGET_COL]

missing = [c for c in required_cols if c not in dfp.columns]

if missing:
    raise ValueError(
        "Faltan columnas en df_daily.xlsx: "
        + ", ".join(missing)
        + ". Columnas disponibles (primeras 40): "
        + ", ".join(list(dfp.columns)[:40])
    )

print("OK dfp listo.")
print("Rango:", dfp.index.min(), "->", dfp.index.max())
print("Columnas:", list(dfp.columns))


OK dfp listo.
Rango: 2015-01-05 00:00:00 -> 2025-09-05 00:00:00
Columnas: ['compras_ANCAP', 'int_ANCAP', 'var_stock_pet', 'inv_ANCAP', 'venta_autos', 'saldo_cuenta_corriente', 'ingreso_primario', 'inversion_directa', 'inversion_de_cartera', 'derivados_financieros', 'adq_neta_act_financieros', 'act_ext_sist_brio', 'cred_int_publ_sist_brio', 'cred_neto_al_BCU', 'cred_priv_res_sist_brio', 'LRM_sist_brio', 'act_ext_BCU', 'cred_int_publ_BCU', 'cred_neto_al_sist_brio', 'cred_priv_res_BCU', 'LRM_res_BCU', 'dep_plazo_MN', 'dep_ME', 'TPM_br', 'usd_br', 'IPC_br', 'IA_br', 'inflacion_12m_br', 'EPU_br', 'total_deuda_publica', 'dolar_blue_arg', 'dolar', 'exp_agr_y_gan', 'exp_carne', 'exp_arroz', 'exp_madera', 'exp_veh_y_trans', 'exp_quimicos', 'exp_arg', 'exp_brl', 'exp_eeuu', 'exp_EU', 'exp_chn', 'exp_inflacion', 'VIX', 'WTI', 'IPC_usa', 'core_pce_USA', 'FedFunds', 'NFCI', 'tasa_desempleo_usa', 'M2_USA', 'USD_NominalBroad', 'IVA', 'IMESI', 'IRPF', 'IRAE', 'imp_patrimonio', 'ingresos_gob', 'rem_gob

## 4) Helpers: target por horizonte + métricas en niveles + conversión retornos→niveles


In [5]:
def future_sum(series: pd.Series, H: int) -> pd.Series:
    """Retorno log acumulado desde t+1 hasta t+H, alineado en t."""
    s = series.astype(float)
    out = pd.Series(np.nan, index=s.index, dtype=float)
    out[:] = s.shift(-1).rolling(window=H).sum().shift(-(H-1))
    return out

def compute_metrics_levels(y_true_level, y_pred_level):
    yt = pd.to_numeric(pd.Series(y_true_level), errors="coerce")
    yp = pd.to_numeric(pd.Series(y_pred_level), errors="coerce")
    m = pd.concat([yt, yp], axis=1).dropna()
    if len(m) == 0:
        return {"MAE_level": np.nan, "RMSE_level": np.nan, "R2_level": np.nan}

    yt2 = m.iloc[:, 0].values
    yp2 = m.iloc[:, 1].values
    return {
        "MAE_level": float(mean_absolute_error(yt2, yp2)),
        "RMSE_level": float(np.sqrt(mean_squared_error(yt2, yp2))),
        "R2_level": float(r2_score(yt2, yp2)) if len(yt2) > 1 else np.nan,
    }

def returns_oos_to_levels_df(dt_oos, yhat_returns_oos, H: int, dfp: pd.DataFrame) -> pd.DataFrame:
    """Convierte predicciones OOS en retornos acumulados (log) a NIVELES.

    Devuelve un dataframe con estructura estándar (en niveles):
        fecha, fecha_objetivo, dolar_t, dolar_pred_tH, dolar_real_tH
    """
    idx = pd.to_datetime(pd.Series(dt_oos))

    # Spot en t
    dolar_t = pd.to_numeric(dfp["dolar"].reindex(idx), errors="coerce")

    # Fecha objetivo y dólar observado en t+H (H filas adelante en la grilla de dfp)
    idx_series = pd.Series(dfp.index, index=dfp.index)
    fecha_objetivo = pd.to_datetime(idx_series.shift(-H).reindex(idx))
    dolar_real_tH = pd.to_numeric(dfp["dolar"].shift(-H).reindex(idx), errors="coerce")

    # Predicción en nivel: S_{t+H} = S_t * exp(ŷ_ret_acum)
    yhat = pd.to_numeric(pd.Series(yhat_returns_oos), errors="coerce").values
    dolar_pred_tH = dolar_t.values * np.exp(yhat)

    out = pd.DataFrame({
        "fecha": idx,
        "fecha_objetivo": fecha_objetivo.values,
        "dolar_t": dolar_t.values,
        "dolar_pred_tH": dolar_pred_tH,
        "dolar_real_tH": dolar_real_tH.values,
    })

    out = out.dropna().sort_values("fecha").reset_index(drop=True)
    return out


## 5) Profiling por modelo (tiempo + memoria)


In [6]:
import psutil
_PROC = psutil.Process(os.getpid())

def _fmt_seconds(s: float) -> str:
    if s < 60:
        return f"{s:.2f}s"
    if s < 3600:
        return f"{s/60:.2f}m"
    return f"{s/3600:.2f}h"

def _mem_mb() -> float:
    return _PROC.memory_info().rss / (1024**2)

def profile_start(model_name: str):
    return {"model": model_name, "t0": time.perf_counter(), "mem0": _mem_mb()}

def profile_end(p, extra: dict | None = None):
    elapsed = time.perf_counter() - p["t0"]
    mem1 = _mem_mb()
    out = {
        "model": p["model"],
        "elapsed": _fmt_seconds(elapsed),
        "mem_delta_mb": round(mem1 - p["mem0"], 2),
        "mem_now_mb": round(mem1, 2),
    }
    if extra:
        out.update(extra)
    print(f"[PROFILE] {out}")
    return out


## 6) Cache por horizonte (arma `y` y `dates` una sola vez)


In [7]:
cache_by_h = {}

df_work = df_daily[[DATE_COL, TARGET_COL]].copy()
df_work[DATE_COL] = pd.to_datetime(df_work[DATE_COL], errors="coerce")
df_work[TARGET_COL] = pd.to_numeric(df_work[TARGET_COL], errors="coerce")
df_work = (
    df_work
    .dropna(subset=[DATE_COL, TARGET_COL])
    .sort_values(DATE_COL)
    .reset_index(drop=True)
)

for h_tag, H in HORIZONS.items():
    # Target multi-horizonte: retorno log acumulado futuro (alineado en t)
    y = future_sum(df_work[TARGET_COL], H)
    valid = ~y.isna()
    yv = y.loc[valid].reset_index(drop=True)
    dv = df_work.loc[valid, DATE_COL].reset_index(drop=True)

    cache_by_h[h_tag] = {"H": H, "y": yv, "dates": dv}

    print(f"[CACHE] {h_tag}: H={H} | n={len(yv)} | {dv.min()} -> {dv.max()}")

[CACHE] 1m: H=21 | n=2622 | 2015-01-05 00:00:00 -> 2025-08-06 00:00:00
[CACHE] 3m: H=63 | n=2580 | 2015-01-05 00:00:00 -> 2025-06-05 00:00:00
[CACHE] 6m: H=126 | n=2517 | 2015-01-05 00:00:00 -> 2025-03-05 00:00:00
[CACHE] 12m: H=252 | n=2391 | 2015-01-05 00:00:00 -> 2024-09-02 00:00:00


## 7) Preparar paridades RPPP y CIRP


In [8]:
# ===============================
# RPPP y CIRP — construcción de Shat
# ===============================

def _find_col(cols, candidates):
    norm = {str(c).strip().lower(): c for c in cols}
    for cand in candidates:
        key = cand.strip().lower()
        if key in norm:
            return norm[key]
    for c in cols:
        c2 = str(c).strip().lower()
        if any(cand.strip().lower() in c2 for cand in candidates):
            return c
    return None

def _to_num_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace("%", "", regex=False)
    s = s.str.replace(".", "", regex=False).where(s.str.contains(","), s)
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

def _month_end_index(s: pd.Series) -> pd.Series:
    dt = pd.to_datetime(s, errors="coerce")
    return dt.dt.to_period("M").dt.to_timestamp("M")

def _normalize_rate_series(s: pd.Series, assume_percent_if_gt1: bool = True) -> pd.Series:
    out = _to_num_series(s)
    med = out.dropna().median()
    if assume_percent_if_gt1 and pd.notna(med) and med > 1:
        out = out / 100.0
    return out

def _roll_to_next_business_day(ts: pd.Timestamp) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    while ts.weekday() >= 5:
        ts = ts + pd.Timedelta(days=1)
    return ts

def _approx_release_date(month_end: pd.Timestamp, release_day: int) -> pd.Timestamp:
    month_end = pd.Timestamp(month_end)
    release_month_start = month_end + pd.offsets.MonthBegin(1)
    approx = release_month_start + pd.Timedelta(days=release_day - 1)
    return _roll_to_next_business_day(approx)


In [9]:
# ----------- Cargar IPC para RPPP -----------
ipc_uy = pd.read_excel(INPUT_IPC_UY_FILE).copy()
ipc_us = pd.read_excel(INPUT_IPC_US_FILE).copy()

ipc_uy.columns = [str(c).strip() for c in ipc_uy.columns]
ipc_us.columns = [str(c).strip() for c in ipc_us.columns]

uy_date_col = _find_col(ipc_uy.columns, ["Fecha", "date", "DATE"])
if uy_date_col is None:
    raise ValueError(f"No se encontró columna de fecha en {INPUT_IPC_UY_FILE}. Columnas: {list(ipc_uy.columns)}")

us_date_col = _find_col(ipc_us.columns, ["DATE", "Fecha", "date"])
if us_date_col is None:
    raise ValueError(f"No se encontró columna de fecha en {INPUT_IPC_US_FILE}. Columnas: {list(ipc_us.columns)}")

uy_value_candidates = [c for c in ipc_uy.columns if c != uy_date_col]
us_value_candidates = [c for c in ipc_us.columns if c != us_date_col]

if not uy_value_candidates:
    raise ValueError(f"No se encontró columna de valor en {INPUT_IPC_UY_FILE}.")
if not us_value_candidates:
    raise ValueError(f"No se encontró columna de valor en {INPUT_IPC_US_FILE}.")

uy_value_col = uy_value_candidates[0]
us_value_col = us_value_candidates[0]

ipc_uy[uy_date_col] = _month_end_index(ipc_uy[uy_date_col])
ipc_us[us_date_col] = _month_end_index(ipc_us[us_date_col])

ipc_uy = (
    ipc_uy
    .dropna(subset=[uy_date_col])
    .sort_values(uy_date_col)
    .drop_duplicates(subset=[uy_date_col], keep="last")
    .set_index(uy_date_col)
)

ipc_us = (
    ipc_us
    .dropna(subset=[us_date_col])
    .sort_values(us_date_col)
    .drop_duplicates(subset=[us_date_col], keep="last")
    .set_index(us_date_col)
)

ipc_uy_val = _to_num_series(ipc_uy[uy_value_col]).rename("ipc_uy")
ipc_us_val = _to_num_series(ipc_us[us_value_col]).rename("ipc_us")

ipc_monthly = pd.concat([ipc_uy_val, ipc_us_val], axis=1).sort_index()
ipc_monthly = ipc_monthly.dropna(subset=["ipc_uy", "ipc_us"]).copy()
ipc_monthly.index.name = "month_ref"

# Inflación mensual simple a partir del IPC
ipc_monthly["pi_uy_m"] = ipc_monthly["ipc_uy"].pct_change()
ipc_monthly["pi_us_m"] = ipc_monthly["ipc_us"].pct_change()

# Fechas efectivas de disponibilidad para evitar look-ahead
ipc_monthly["avail_uy"] = ipc_monthly.index.to_series().apply(lambda x: _approx_release_date(x, UY_CPI_RELEASE_DAY))
ipc_monthly["avail_us"] = ipc_monthly.index.to_series().apply(lambda x: _approx_release_date(x, US_CPI_RELEASE_DAY))
ipc_monthly["avail_max"] = ipc_monthly[["avail_uy", "avail_us"]].max(axis=1)

# Inflación acumulada por horizonte directamente desde niveles de IPC:
#   pi_{t,h} = IPC_t / IPC_{t-k} - 1
# Esto es consistente con:
#   Shat_{t+h|t} = S_t * (1 + pi_UY_{t,h}) / (1 + pi_USA_{t,h})
rppp_signal_m = pd.DataFrame(index=ipc_monthly.index)
for h_tag, k_months in HORIZON_MONTHS.items():
    rppp_signal_m[f"pi_uy_{h_tag}"] = ipc_monthly["ipc_uy"] / ipc_monthly["ipc_uy"].shift(k_months) - 1.0
    rppp_signal_m[f"pi_us_{h_tag}"] = ipc_monthly["ipc_us"] / ipc_monthly["ipc_us"].shift(k_months) - 1.0

# Llevar la señal mensual al calendario diario usando la fecha efectiva de disponibilidad
rppp_signal_release = (
    pd.concat([ipc_monthly[["avail_max"]], rppp_signal_m], axis=1)
    .dropna(subset=["avail_max"])
    .sort_values("avail_max")
    .drop_duplicates(subset=["avail_max"], keep="last")
    .set_index("avail_max")
)

rppp_signal_daily = (
    rppp_signal_release[
        [
            "pi_uy_1m", "pi_us_1m",
            "pi_uy_3m", "pi_us_3m",
            "pi_uy_6m", "pi_us_6m",
            "pi_uy_12m", "pi_us_12m",
        ]
    ]
    .reindex(dfp.index, method="ffill")
)

S_t = pd.to_numeric(dfp["dolar"], errors="coerce")

rppp_Shat = pd.DataFrame(index=dfp.index)
for h_tag in HORIZON_MONTHS.keys():
    rppp_Shat[h_tag] = S_t * (
        (1.0 + rppp_signal_daily[f"pi_uy_{h_tag}"]) /
        (1.0 + rppp_signal_daily[f"pi_us_{h_tag}"])
    )


In [10]:
# ----------- Cargar tasas para CIRP -----------
ust = pd.read_excel(INPUT_UST_FILE).copy()
ust.columns = [str(c).strip() for c in ust.columns]

ust_date_col = _find_col(ust.columns, ["date", "Fecha", "DATE"])
if ust_date_col is None:
    raise ValueError(f"No se encontró columna de fecha en {INPUT_UST_FILE}. Columnas: {list(ust.columns)}")

ust[ust_date_col] = pd.to_datetime(ust[ust_date_col], errors="coerce")
ust = (
    ust
    .dropna(subset=[ust_date_col])
    .sort_values(ust_date_col)
    .drop_duplicates(subset=[ust_date_col], keep="last")
    .set_index(ust_date_col)
)

required_ust = ["ust_1m", "ust_3m", "ust_6m", "ust_12m"]
missing_ust = [c for c in required_ust if c not in ust.columns]
if missing_ust:
    raise ValueError(f"Faltan columnas en {INPUT_UST_FILE}: {missing_ust}. Columnas disponibles: {list(ust.columns)}")

for c in required_ust:
    ust[c] = _normalize_rate_series(ust[c], assume_percent_if_gt1=True)

lrm = pd.read_excel(INPUT_LRM_FILE).copy()
lrm.columns = [str(c).strip() for c in lrm.columns]

lrm_date_col = _find_col(lrm.columns, ["Fecha", "date", "DATE"])
if lrm_date_col is None:
    raise ValueError(f"No se encontró columna de fecha en {INPUT_LRM_FILE}. Columnas: {list(lrm.columns)}")

lrm[lrm_date_col] = pd.to_datetime(lrm[lrm_date_col], errors="coerce")
lrm = (
    lrm
    .dropna(subset=[lrm_date_col])
    .sort_values(lrm_date_col)
    .drop_duplicates(subset=[lrm_date_col], keep="last")
    .set_index(lrm_date_col)
)

# Normalizar nombres de columnas de plazos
rename_lrm = {}
for c in lrm.columns:
    raw = str(c).strip().replace(".0", "")
    try:
        raw_num = str(int(float(raw)))
    except Exception:
        raw_num = raw
    if raw_num in {"30", "90", "180", "360", "720"}:
        rename_lrm[c] = raw_num

lrm = lrm.rename(columns=rename_lrm)

required_lrm = ["30", "90", "180", "360"]
missing_lrm = [c for c in required_lrm if c not in lrm.columns]
if missing_lrm:
    raise ValueError(
        f"Faltan columnas en {INPUT_LRM_FILE}: {missing_lrm}. "
        f"Para que CIRP coincida con la metodología se requieren exactamente 30, 90, 180 y 360 días. "
        f"Columnas disponibles: {list(lrm.columns)}"
    )

for c in [col for col in ["30", "90", "180", "360", "720"] if col in lrm.columns]:
    # LRM debería venir en decimal (ej. 0.0579). Si viniera en porcentaje, se corrige automáticamente.
    lrm[c] = _normalize_rate_series(lrm[c], assume_percent_if_gt1=True)

CIRP_MAP = {
    "1m":  ("30",  "ust_1m"),
    "3m":  ("90",  "ust_3m"),
    "6m":  ("180", "ust_6m"),
    "12m": ("360", "ust_12m"),
}

cirp_Shat = {}

# Tau contractual según plazo financiero (no H/252)
TAU_MAP = {
    "1m": 30/360,
    "3m": 90/360,
    "6m": 180/360,
    "12m": 360/360,
}

for h_tag, H in HORIZONS.items():
    uy_col, us_col = CIRP_MAP[h_tag]

    # rellenar huecos internos y luego alinear al calendario diario
    i_uy = lrm[uy_col].sort_index().ffill().reindex(dfp.index, method="ffill")
    i_us = ust[us_col].sort_index().ffill().reindex(dfp.index, method="ffill")

    tau = TAU_MAP[h_tag]

    cirp_Shat[h_tag] = S_t * (1.0 + i_uy * tau) / (1.0 + i_us * tau)

cirp_Shat = pd.DataFrame(cirp_Shat, index=dfp.index)


In [11]:
print("OK RPPP y CIRP construidos con nuevas bases")
print()
print("RPPP | IPC mensual UY/US:", ipc_monthly.index.min(), "->", ipc_monthly.index.max())
print("RPPP | señal diaria disponible:", rppp_signal_daily.index.min(), "->", rppp_signal_daily.index.max())
print("RPPP | primera disponibilidad efectiva:", rppp_signal_release.index.min())
print("RPPP | última disponibilidad efectiva:", rppp_signal_release.index.max())
print("RPPP | release approx UY day:", UY_CPI_RELEASE_DAY, "| US day:", US_CPI_RELEASE_DAY)
print("UST  |", ust.index.min(), "->", ust.index.max())
print("LRM  |", lrm.index.min(), "->", lrm.index.max())

print()
print("Mapeo CIRP utilizado:")
for h_tag, (uy_col, us_col) in CIRP_MAP.items():
    print(f" - {h_tag}: UY={uy_col} días | US={us_col}")

print()
print("Diagnóstico disponibilidad RPPP (últimos 6 meses):")
display(
    ipc_monthly[["avail_uy", "avail_us", "avail_max", "pi_uy_m", "pi_us_m"]]
    .join(rppp_signal_m.tail(len(ipc_monthly)))
    .tail(6)
)


OK RPPP y CIRP construidos con nuevas bases

RPPP | IPC mensual UY/US: 2014-01-31 00:00:00 -> 2026-02-28 00:00:00
RPPP | señal diaria disponible: 2015-01-05 00:00:00 -> 2025-09-05 00:00:00
RPPP | primera disponibilidad efectiva: 2014-02-14 00:00:00
RPPP | última disponibilidad efectiva: 2026-03-16 00:00:00
RPPP | release approx UY day: 5 | US day: 14
UST  | 2014-01-01 00:00:00 -> 2026-03-12 00:00:00
LRM  | 2019-01-02 00:00:00 -> 2026-03-16 00:00:00

Mapeo CIRP utilizado:
 - 1m: UY=30 días | US=ust_1m
 - 3m: UY=90 días | US=ust_3m
 - 6m: UY=180 días | US=ust_6m
 - 12m: UY=360 días | US=ust_12m

Diagnóstico disponibilidad RPPP (últimos 6 meses):


,avail_uy,avail_us,avail_max,pi_uy_m,pi_us_m,pi_uy_1m,pi_us_1m,pi_uy_3m,pi_us_3m,pi_uy_6m,pi_us_6m,pi_uy_12m,pi_us_12m
month_ref,,,,,,,,,,,,,
2025-08-31,2025-09-05,2025-09-15,2025-09-15,-0.000336,0.003483,-0.000336,0.003483,-0.000677,0.008331,0.009344,0.011299,0.041981,0.029386
2025-09-30,2025-10-06,2025-10-14,2025-10-14,0.004240,0.002951,0.004240,0.002951,0.004445,0.008742,0.007900,0.013947,0.042493,0.030226
2025-11-30,2025-12-05,2025-12-15,2025-12-15,0.005386,0.002523,0.005386,0.002523,0.009309,0.008983,0.010053,0.014864,0.044702,0.029883
2025-12-31,2026-01-05,2026-01-14,2026-01-14,-0.000935,0.002978,-0.000935,0.002978,0.008704,0.008475,0.008021,0.016877,0.039976,0.030023
2026-01-31,2026-02-05,2026-02-16,2026-02-16,0.009176,0.001708,0.009176,0.001708,0.013663,0.007226,0.018169,0.016031,0.045990,0.028287
2026-02-28,2026-03-05,2026-03-16,2026-03-16,0.003497,0.002670,0.003497,0.002670,0.011759,0.007374,0.021177,0.016423,0.038194,0.026646


## 8) Helpers de OOS para benchmarks + gráficos


In [12]:
def rw_oos_baseline(y: pd.Series, dates: pd.Series, initial_train_size: int):
    """
    Benchmark Random Walk:
    - Predicción constante 0 para el retorno acumulado futuro.

    Devuelve:
        y_true, y_pred, dates_true
    """
    y = y.reset_index(drop=True)
    dates = dates.reset_index(drop=True)

    y_true = y.iloc[initial_train_size:].astype(float).values
    y_pred = np.zeros_like(y_true, dtype=float)
    dates_true = dates.iloc[initial_train_size:].values

    return y_true, y_pred, dates_true


def save_oos_plot(oos_df: pd.DataFrame, model_name: str, h_tag: str, plots_dir: str, max_points: int = 1200):
    d = oos_df.copy()
    d["fecha"] = pd.to_datetime(d["fecha"])
    d = d.sort_values("fecha")
    if len(d) > max_points:
        d = d.iloc[-max_points:].reset_index(drop=True)

    plt.figure(figsize=(12, 4))
    plt.plot(d["fecha"], d["dolar_real_tH"], label="Real (S_{t+H})")
    plt.plot(d["fecha"], d["dolar_pred_tH"], label="Predicción")
    plt.title(f"{model_name} — horizonte {h_tag} (niveles USD/UYU)")
    plt.xlabel("fecha (t)")
    plt.ylabel("USD/UYU")
    plt.legend()
    plt.tight_layout()

    safe = model_name.replace("(", "").replace(")", "").replace(" ", "")
    outpath = os.path.join(plots_dir, f"plot_{h_tag}_{safe}.png")
    plt.savefig(outpath, dpi=160)
    plt.close()
    return outpath

def levels_oos_from_parity(dates: pd.Series, H: int, Shat_series: pd.Series, dfp: pd.DataFrame) -> pd.DataFrame:
    """Construye OOS en niveles a partir de una predicción en nivel (paridades, encuestas, etc.).

    Devuelve un dataframe con estructura estándar:
        fecha, fecha_objetivo, dolar_t, dolar_pred_tH, dolar_real_tH
    """
    idx = pd.to_datetime(pd.Series(dates))

    dolar_t = pd.to_numeric(dfp["dolar"].reindex(idx), errors="coerce")

    idx_series = pd.Series(dfp.index, index=dfp.index)
    fecha_objetivo = pd.to_datetime(idx_series.shift(-H).reindex(idx))
    dolar_real_tH = pd.to_numeric(dfp["dolar"].shift(-H).reindex(idx), errors="coerce")

    dolar_pred_tH = pd.to_numeric(Shat_series.reindex(idx), errors="coerce")

    out = pd.DataFrame({
        "fecha": idx,
        "fecha_objetivo": fecha_objetivo.values,
        "dolar_t": dolar_t.values,
        "dolar_pred_tH": dolar_pred_tH.values,
        "dolar_real_tH": dolar_real_tH.values,
    })

    out = out.dropna().sort_values("fecha").reset_index(drop=True)
    return out


def build_common_sample_metrics(oos_store: dict) -> pd.DataFrame:
    rows = []
    all_h = sorted(set(h for h, _ in oos_store.keys()), key=lambda x: list(HORIZONS.keys()).index(x) if x in HORIZONS else x)

    for h_tag in all_h:
        model_oos = {}
        for (h, model_name), oos in oos_store.items():
            if h != h_tag or oos is None or oos.empty:
                continue
            d = oos.copy()
            d["fecha"] = pd.to_datetime(d["fecha"])
            d = d.dropna(subset=["fecha", "dolar_real_tH", "dolar_pred_tH"]).sort_values("fecha")
            if not d.empty:
                model_oos[model_name] = d

        if len(model_oos) < 2:
            continue

        common_dates = None
        for d in model_oos.values():
            s = set(d["fecha"])
            common_dates = s if common_dates is None else common_dates.intersection(s)

        common_dates = sorted(common_dates) if common_dates else []
        if not common_dates:
            continue

        common_dates_idx = pd.DatetimeIndex(common_dates)

        for model_name, d in model_oos.items():
            d_common = d[d["fecha"].isin(common_dates_idx)].copy().sort_values("fecha")
            met = compute_metrics_levels(d_common["dolar_real_tH"].values, d_common["dolar_pred_tH"].values)
            rows.append({
                "h_tag": h_tag,
                "H_days": HORIZONS.get(h_tag, np.nan),
                "model": model_name,
                **met,
                "n_common": len(d_common),
                "common_start": d_common["fecha"].min(),
                "common_end": d_common["fecha"].max(),
                "params": "metricas_recalculadas_en_muestra_comun"
            })

    if not rows:
        return pd.DataFrame(columns=["h_tag", "H_days", "model", "MAE_level", "RMSE_level", "R2_level", "n_common", "common_start", "common_end", "params"])

    return (
        pd.DataFrame(rows)
        .sort_values(["h_tag", "MAE_level", "model"])
        .reset_index(drop=True)
    )


In [13]:
# ===============================
# Ventana de comparación con ML/DL
# ===============================
ML_COMPARISON_START = pd.Timestamp("2022-06-01")
print("ML/DL comparison window starts:", ML_COMPARISON_START.date())

ML/DL comparison window starts: 2022-06-01


## 8) Estructuras de resultados


In [14]:
results_rows = []
oos_store = {}
profiles = []

# 9) Benchmark: Random Walk

**Random Walk**

Este benchmark corresponde a un Random Walk, bajo el supuesto de que el mejor predictor del tipo de cambio futuro es su valor actual, sin incorporar información adicional ni dinámica estructural. En este marco, las variaciones del tipo de cambio son impredecibles y siguen un proceso estocástico sin deriva. El Random Walk se utiliza como benchmark ingenuo estándar en la literatura, sin estimación de parámetros ni entrenamiento, y constituye una referencia mínima contra la cual evaluar el desempeño de modelos más complejos.


In [15]:
MODEL_NAME = "RandomWalk"
p = profile_start(MODEL_NAME)

total_oos = 0
for h_tag, H in HORIZONS.items():
    y = cache_by_h[h_tag]["y"]
    dates = cache_by_h[h_tag]["dates"]

    # Sin split train/test: usamos todas las observaciones evaluables
    yt_ret, yp_ret, dt = rw_oos_baseline(y, dates, initial_train_size=0)
    oos = returns_oos_to_levels_df(dt, yp_ret, H=H, dfp=dfp)
    total_oos += len(oos)

    met = compute_metrics_levels(oos["dolar_real_tH"].values, oos["dolar_pred_tH"].values)
    results_rows.append({"h_tag": h_tag, "H_days": H, "model": MODEL_NAME, **met, "params": ""})
    oos_store[(h_tag, MODEL_NAME)] = oos

    save_oos_plot(oos, MODEL_NAME, h_tag, PLOTS_DIR)

profiles.append(profile_end(p, extra={"total_oos_rows": total_oos}))
print("OK", MODEL_NAME)

[PROFILE] {'model': 'RandomWalk', 'elapsed': '0.99s', 'mem_delta_mb': 13.35, 'mem_now_mb': 252.3, 'total_oos_rows': 10110}
OK RandomWalk


# 10) Benchmark: RPPP

**RPPP — Relative Purchasing Power Parity**

Este benchmark se fundamenta en la Paridad del Poder de Compra Relativa (RPPP), según la cual las variaciones del tipo de cambio entre dos países reflejan, en el largo plazo, las diferencias en sus tasas de inflación. En esta versión se implementa con **IPC mensual de Uruguay y Estados Unidos**, inflación acumulada por horizonte (**1, 3, 6 y 12 meses**) construida **directamente desde los niveles del IPC**, y una corrección explícita por **fecha efectiva de disponibilidad** del dato para evitar *look-ahead bias*.

In [16]:
MODEL_NAME = "RPPP"
p = profile_start(MODEL_NAME)

total_oos = 0
for h_tag, H in HORIZONS.items():
    dates = cache_by_h[h_tag]["dates"]
    dt_oos = dates.reset_index(drop=True)  # sin split train/test


    oos = levels_oos_from_parity(dt_oos, H=H, Shat_series=rppp_Shat[h_tag], dfp=dfp)
    total_oos += len(oos)

    met = compute_metrics_levels(oos["dolar_real_tH"].values, oos["dolar_pred_tH"].values)
    results_rows.append({"h_tag": h_tag, "H_days": H, "model": MODEL_NAME, **met, "params": ""})
    oos_store[(h_tag, MODEL_NAME)] = oos

    save_oos_plot(oos, MODEL_NAME, h_tag, PLOTS_DIR)

profiles.append(profile_end(p, extra={"total_oos_rows": total_oos}))
print("OK", MODEL_NAME)

[PROFILE] {'model': 'RPPP', 'elapsed': '0.98s', 'mem_delta_mb': 0.89, 'mem_now_mb': 253.2, 'total_oos_rows': 10081}
OK RPPP


# 11) Benchmark: CIRP

**CIRP — Covered Interest Rate Parity**

Este benchmark se basa en la Paridad Cubierta de Tasas de Interés (CIRP), que establece que, en ausencia de arbitraje, la rentabilidad de invertir en moneda local cubierta con un contrato forward debe ser equivalente a la de invertir en moneda extranjera. Bajo este principio, el tipo de cambio forward implícito surge de la relación entre las tasas de interés domésticas y externas. En este trabajo, la CIRP se utiliza como benchmark teórico de referencia, sin estimación de parámetros ni entrenamiento, aplicando directamente la relación de paridad para generar predicciones del tipo de cambio a distintos horizontes.


In [17]:
MODEL_NAME = "CIRP"
p = profile_start(MODEL_NAME)

total_oos = 0
for h_tag, H in HORIZONS.items():
    dates = cache_by_h[h_tag]["dates"]
    dt_oos = dates.reset_index(drop=True)  # sin split train/test


    oos = levels_oos_from_parity(dt_oos, H=H, Shat_series=cirp_Shat[h_tag], dfp=dfp)
    total_oos += len(oos)

    met = compute_metrics_levels(oos["dolar_real_tH"].values, oos["dolar_pred_tH"].values)
    results_rows.append({"h_tag": h_tag, "H_days": H, "model": MODEL_NAME, **met, "params": ""})
    oos_store[(h_tag, MODEL_NAME)] = oos

    save_oos_plot(oos, MODEL_NAME, h_tag, PLOTS_DIR)

profiles.append(profile_end(p, extra={"total_oos_rows": total_oos}))
print("OK", MODEL_NAME)

[PROFILE] {'model': 'CIRP', 'elapsed': '1.15s', 'mem_delta_mb': 7.8, 'mem_now_mb': 261.0, 'total_oos_rows': 6175}
OK CIRP


# 12) Benchmark: encuestas_BCU


In [18]:
MODEL_NAME = "encuestas_BCU"
p = profile_start(MODEL_NAME)

# -------------------------
# 1) Serie REAL mensual: dólar al cierre de mes (desde df_daily.xlsx)
# -------------------------
# dfp ya existe: index=Fecha (daily), col "dolar"
# Creamos una serie mensual al cierre (MonthEnd)
dolar_me = (
    dfp["dolar"]
    .resample("M")
    .last()
    .dropna()
)

# -------------------------
# 2) Cargar encuestas BCU (mensual, en niveles)
# -------------------------
df_bcu = pd.read_excel(INPUT_BCU_FILE).copy()
df_bcu.columns = [c.strip() for c in df_bcu.columns]

rename_map = {
    "Fecha": "Fecha",
    "1 mes": "bcu_1m",
    "6 meses": "bcu_6m",
    "12 meses": "bcu_12m",
}

missing_cols = [c for c in rename_map.keys() if c not in df_bcu.columns]
if missing_cols:
    raise ValueError(
        f"Faltan columnas en {INPUT_BCU_FILE}: {missing_cols}. "
        f"Columnas encontradas: {list(df_bcu.columns)}"
    )

df_bcu = df_bcu.rename(columns=rename_map)

# Fecha -> llevarla a cierre de mes
df_bcu["Fecha"] = pd.to_datetime(df_bcu["Fecha"], errors="coerce")
df_bcu = df_bcu.dropna(subset=["Fecha"]).sort_values("Fecha")
df_bcu["Fecha"] = df_bcu["Fecha"].dt.to_period("M").dt.to_timestamp("M")

# Nos quedamos solo con meses donde tenemos REAL
df_bcu = df_bcu[df_bcu["Fecha"].isin(dolar_me.index)].reset_index(drop=True)

display(df_bcu.head())


# -------------------------
# 3) Helper: construir OOS mensual en estructura estándar (niveles)
# -------------------------
def oos_from_bcu_monthly_standard(df_bcu_in: pd.DataFrame,
                                  dolar_me: pd.Series,
                                  k_months: int,
                                  pred_col: str) -> pd.DataFrame:

    d = df_bcu_in[["Fecha", pred_col]].copy()
    d[pred_col] = pd.to_numeric(d[pred_col], errors="coerce")

    fecha = pd.to_datetime(d["Fecha"])

    # Fecha objetivo correctamente construida
    fecha_objetivo = fecha + pd.offsets.MonthEnd(k_months)

    # Nivel observado en t
    dolar_t = dolar_me.reindex(fecha)

    # Nivel real en t+k
    dolar_real_tH = dolar_me.shift(-k_months).reindex(fecha)

    # Predicción en nivel ya viene directa
    dolar_pred_tH = d[pred_col].astype(float)

    out = pd.DataFrame({
        "fecha": fecha,
        "fecha_objetivo": fecha_objetivo,
        "dolar_t": dolar_t.values,
        "dolar_pred_tH": dolar_pred_tH.values,
        "dolar_real_tH": dolar_real_tH.values,
    })

    out = (
        out
        .dropna()
        .sort_values("fecha")
        .reset_index(drop=True)
    )

    return out


# -------------------------
# 4) Correr horizons disponibles
# -------------------------
BCU_H = {
    "1m": (1, "bcu_1m"),
    "6m": (6, "bcu_6m"),
    "12m": (12, "bcu_12m"),
}

total_oos = 0
extra_log = []

for h_tag, (k_months, pred_col) in BCU_H.items():

    oos = oos_from_bcu_monthly_standard(
        df_bcu_in=df_bcu,
        dolar_me=dolar_me,
        k_months=k_months,
        pred_col=pred_col
    )

    extra_log.append({
        "h_tag": h_tag,
        "k_months": k_months,
        "oos_rows": len(oos)
    })

    total_oos += len(oos)

    met = compute_metrics_levels(
        oos["dolar_real_tH"].values,
        oos["dolar_pred_tH"].values
    )

    H_days = HORIZONS.get(h_tag, np.nan)

    results_rows.append({
        "h_tag": h_tag,
        "H_days": H_days,
        "model": MODEL_NAME,
        **met,
        "params": f"mensual_me; k_months={k_months}"
    })

    oos_store[(h_tag, MODEL_NAME)] = oos

    save_oos_plot(oos, MODEL_NAME, h_tag, PLOTS_DIR)


profiles.append(
    profile_end(
        p,
        extra={
            "total_oos_rows": total_oos,
            "detail": extra_log
        }
    )
)

print("OK", MODEL_NAME)
display(pd.DataFrame(extra_log))


,Fecha,bcu_1m,bcu_6m,bcu_12m
0,2018-01-31,28.63,29.60,30.83
1,2018-02-28,28.55,29.50,30.80
2,2018-03-31,28.50,29.60,30.84
3,2018-04-30,28.35,29.45,30.65
4,2018-05-31,30.30,30.92,31.80


[PROFILE] {'model': 'encuestas_BCU', 'elapsed': '0.88s', 'mem_delta_mb': 2.57, 'mem_now_mb': 263.62, 'total_oos_rows': 260, 'detail': [{'h_tag': '1m', 'k_months': 1, 'oos_rows': 92}, {'h_tag': '6m', 'k_months': 6, 'oos_rows': 87}, {'h_tag': '12m', 'k_months': 12, 'oos_rows': 81}]}
OK encuestas_BCU


,h_tag,k_months,oos_rows
0,1m,1,92
1,6m,6,87
2,12m,12,81


## 13) Consolidación y export


In [19]:
results_df = (
    pd.DataFrame(results_rows)
    .drop_duplicates(subset=["h_tag", "model"], keep="last")
    .sort_values(["h_tag", "MAE_level", "model"])
    .reset_index(drop=True)
)
profiles_df = pd.DataFrame(profiles)
results_common_df = build_common_sample_metrics(oos_store)

display(results_df)
display(results_common_df)
display(profiles_df)

# -------------------------
# Export: resumen + profiling (CSV y XLSX)
# -------------------------
results_csv   = os.path.join(BASE_SAVE_DIR, "fx_results_benchmarks.csv")
results_xlsx  = os.path.join(BASE_SAVE_DIR, "fx_results_benchmarks.xlsx")
results_common_csv  = os.path.join(BASE_SAVE_DIR, "fx_results_benchmarks_common_sample.csv")
results_common_xlsx = os.path.join(BASE_SAVE_DIR, "fx_results_benchmarks_common_sample.xlsx")
profiles_csv  = os.path.join(BASE_SAVE_DIR, "profiles_benchmarks.csv")
profiles_xlsx = os.path.join(BASE_SAVE_DIR, "profiles_benchmarks.xlsx")

results_df.to_csv(results_csv, index=False)
results_df.to_excel(results_xlsx, index=False)
results_common_df.to_csv(results_common_csv, index=False)
results_common_df.to_excel(results_common_xlsx, index=False)
profiles_df.to_csv(profiles_csv, index=False)
profiles_df.to_excel(profiles_xlsx, index=False)

# -------------------------
# Export: OOS por modelo/horizonte (CSV y XLSX)
# Estructura estándar:
#   fecha, fecha_objetivo, dolar_t, dolar_pred_tH, dolar_real_tH
# -------------------------
for (h_tag, model_name), oos in oos_store.items():
    safe = model_name.replace("(", "").replace(")", "").replace(" ", "")
    path_csv  = os.path.join(OOS_DIR, f"oos_{h_tag}_{safe}.csv")
    path_xlsx = os.path.join(OOS_DIR, f"oos_{h_tag}_{safe}.xlsx")

    oos.to_csv(path_csv, index=False)
    oos.to_excel(path_xlsx, index=False)

print("Guardado:")
print(" -", results_csv)
print(" -", results_xlsx)
print(" -", results_common_csv)
print(" -", results_common_xlsx)
print(" -", profiles_csv)
print(" -", profiles_xlsx)
print("OOS guardados en:", OOS_DIR, "| Cantidad:", len(os.listdir(OOS_DIR)))
print("Plots guardados en:", PLOTS_DIR, "| Cantidad:", len(os.listdir(PLOTS_DIR)))


,h_tag,H_days,model,MAE_level,RMSE_level,R2_level,params
0,12m,252,RandomWalk,3.020142,3.763407,0.532111,
1,12m,252,RPPP,3.087984,3.504459,0.593661,
2,12m,252,encuestas_BCU,3.239630,3.747377,-0.716184,mensual_me; k_months=12
3,12m,252,CIRP,3.714886,4.306648,-3.322242,
4,1m,21,RandomWalk,0.624024,0.937946,0.974871,
5,1m,21,RPPP,0.648299,0.956114,0.973888,
6,1m,21,CIRP,0.711394,1.060673,0.859160,
7,1m,21,encuestas_BCU,0.910435,1.190459,0.914304,mensual_me; k_months=1
8,3m,63,RandomWalk,1.302473,1.705211,0.913164,
9,3m,63,RPPP,1.352542,1.738183,0.909773,


,h_tag,H_days,model,MAE_level,RMSE_level,R2_level,n_common,common_start,common_end,params
0,12m,252,RandomWalk,2.936364,3.769612,-2.913317,44,2019-01-31,2024-07-31,metricas_recalculadas_en_muestra_comun
1,12m,252,RPPP,3.035322,3.512873,-2.398417,44,2019-01-31,2024-07-31,metricas_recalculadas_en_muestra_comun
2,12m,252,encuestas_BCU,3.451591,3.976472,-2.872777,44,2019-01-31,2024-07-31,metricas_recalculadas_en_muestra_comun
3,12m,252,CIRP,3.673766,4.216459,-3.896068,44,2019-01-31,2024-07-31,metricas_recalculadas_en_muestra_comun
4,1m,21,RandomWalk,0.575192,0.811545,0.911040,52,2019-01-31,2025-07-31,metricas_recalculadas_en_muestra_comun
5,1m,21,CIRP,0.582636,0.812850,0.910754,52,2019-01-31,2025-07-31,metricas_recalculadas_en_muestra_comun
6,1m,21,RPPP,0.623197,0.837548,0.905248,52,2019-01-31,2025-07-31,metricas_recalculadas_en_muestra_comun
7,1m,21,encuestas_BCU,0.840769,1.027862,0.862300,52,2019-01-31,2025-07-31,metricas_recalculadas_en_muestra_comun
8,3m,63,RandomWalk,1.396080,1.854606,0.485379,1597,2019-01-02,2025-06-05,metricas_recalculadas_en_muestra_comun
9,3m,63,RPPP,1.523917,1.919436,0.448771,1597,2019-01-02,2025-06-05,metricas_recalculadas_en_muestra_comun


,model,elapsed,mem_delta_mb,mem_now_mb,total_oos_rows,detail
0,RandomWalk,0.99s,13.35,252.30,10110,NaN
1,RPPP,0.98s,0.89,253.20,10081,NaN
2,CIRP,1.15s,7.80,261.00,6175,NaN
3,encuestas_BCU,0.88s,2.57,263.62,260,"[{'h_tag': '1m', 'k_months': 1, 'oos_rows': 92..."


Guardado:
 - results_n03\fx_results_benchmarks.csv
 - results_n03\fx_results_benchmarks.xlsx
 - results_n03\fx_results_benchmarks_common_sample.csv
 - results_n03\fx_results_benchmarks_common_sample.xlsx
 - results_n03\profiles_benchmarks.csv
 - results_n03\profiles_benchmarks.xlsx
OOS guardados en: results_n03\oos | Cantidad: 46
Plots guardados en: results_n03\plots | Cantidad: 23


In [20]:
# ===============================
# Re-plot con ventana común por horizonte
# ===============================

# 1) Calcular inicio común por horizonte (máximo de fechas mínimas)
common_start_by_h = {}

for h_tag in set(k[0] for k in oos_store.keys()):
    fechas_min = [
        oos["fecha"].min()
        for (h, _), oos in oos_store.items()
        if h == h_tag and not oos.empty
    ]
    if fechas_min:
        common_start_by_h[h_tag] = max(fechas_min)

# 2) Re-plot usando ventana común
for (h_tag, model_name), oos in oos_store.items():

    if h_tag not in common_start_by_h:
        continue

    start_date = common_start_by_h[h_tag]

    oos_plot = (
        oos[oos["fecha"] >= start_date]
        .copy()
        .sort_values("fecha")
    )

    if not oos_plot.empty:
        save_oos_plot(oos_plot, model_name, h_tag, PLOTS_DIR)

In [21]:
# ===============================
# Mejor modelo por horizonte (según RMSE)
# ===============================

METRIC = "MAE_level"   # podés cambiar a MAE o MAPE si querés

best_by_h = (
    results_df
    .sort_values(["h_tag", METRIC], ascending=[True, True])
    .groupby("h_tag", as_index=False)
    .first()
    .loc[:, ["h_tag", "model", METRIC]]
)

display(best_by_h)


,h_tag,model,MAE_level
0,12m,RandomWalk,3.020142
1,1m,RandomWalk,0.624024
2,3m,RandomWalk,1.302473
3,6m,RPPP,1.857357


In [22]:
# ===============================
# DataFrames por horizonte
# ===============================

dfs_by_h = {
    h: (
        results_df
        .query("h_tag == @h")
        .sort_values("MAE_level")   # o RMSE si preferís
        .reset_index(drop=True)
    )
    for h in results_df["h_tag"].unique()
}

# Ejemplo de uso
display(dfs_by_h["1m"])

,h_tag,H_days,model,MAE_level,RMSE_level,R2_level,params
0,1m,21,RandomWalk,0.624024,0.937946,0.974871,
1,1m,21,RPPP,0.648299,0.956114,0.973888,
2,1m,21,CIRP,0.711394,1.060673,0.859160,
3,1m,21,encuestas_BCU,0.910435,1.190459,0.914304,mensual_me; k_months=1


In [22]:
display(dfs_by_h["3m"])

,h_tag,H_days,model,MAE_level,RMSE_level,R2_level,params
0,3m,63,RandomWalk,1.302473,1.705211,0.913164,
1,3m,63,RPPP,1.352542,1.738183,0.909773,
2,3m,63,CIRP,1.544635,1.921052,0.447843,


In [23]:
display(dfs_by_h["6m"])

,h_tag,H_days,model,MAE_level,RMSE_level,R2_level,params
0,6m,126,RPPP,1.857357,2.354415,0.826074,
1,6m,126,RandomWalk,1.973417,2.492368,0.805095,
2,6m,126,encuestas_BCU,2.091264,2.511800,0.464444,mensual_me; k_months=6
3,6m,126,CIRP,2.259937,2.763247,-0.381650,


In [24]:
display(dfs_by_h["12m"])

,h_tag,H_days,model,MAE_level,RMSE_level,R2_level,params
0,12m,252,RandomWalk,3.020142,3.763407,0.532111,
1,12m,252,RPPP,3.087984,3.504459,0.593661,
2,12m,252,encuestas_BCU,3.239630,3.747377,-0.716184,mensual_me; k_months=12
3,12m,252,CIRP,3.714886,4.306648,-3.322242,


In [23]:
for (h_tag, model), oos in oos_store.items():
    print(f"{model:20s} | {h_tag} | start={oos['fecha'].min().date()} | n={len(oos)}")

RandomWalk           | 1m | start=2015-01-05 | n=2622
RandomWalk           | 3m | start=2015-01-05 | n=2580
RandomWalk           | 6m | start=2015-01-05 | n=2517
RandomWalk           | 12m | start=2015-01-05 | n=2391
RPPP                 | 1m | start=2015-01-05 | n=2622
RPPP                 | 3m | start=2015-01-05 | n=2580
RPPP                 | 6m | start=2015-01-05 | n=2517
RPPP                 | 12m | start=2015-02-18 | n=2362
CIRP                 | 1m | start=2019-01-02 | n=1639
CIRP                 | 3m | start=2019-01-02 | n=1597
CIRP                 | 6m | start=2019-01-04 | n=1532
CIRP                 | 12m | start=2019-01-03 | n=1407
encuestas_BCU        | 1m | start=2018-01-31 | n=92
encuestas_BCU        | 6m | start=2018-01-31 | n=87
encuestas_BCU        | 12m | start=2018-01-31 | n=81
